In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torch.nn.utils.rnn import pad_sequence
from collections import Counter
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import re

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
from tensorflow.keras.datasets import imdb as keras_imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

VOCAB_SIZE  = 10_000
MAX_LEN     = 256
BATCH_SIZE  = 64

(x_train, y_train), (x_test, y_test) = keras_imdb.load_data(num_words=VOCAB_SIZE)

x_train = pad_sequences(x_train, maxlen=MAX_LEN, padding='post', truncating='post')
x_test  = pad_sequences(x_test,  maxlen=MAX_LEN, padding='post', truncating='post')

print(f"Train: {x_train.shape}, Test: {x_test.shape}")

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train: (25000, 256), Test: (25000, 256)


In [3]:
class IMDBDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):  return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

train_loader = DataLoader(IMDBDataset(x_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(IMDBDataset(x_test,  y_test),  batch_size=BATCH_SIZE)
print("DataLoaders ready.")

DataLoaders ready.


In [4]:
EMBED_DIM  = 100
HIDDEN_DIM = 128
N_LAYERS   = 1

class SentimentModel(nn.Module):
    def __init__(self, rnn_type='RNN'):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB_SIZE, EMBED_DIM, padding_idx=0)

        rnn_cls = {'RNN': nn.RNN, 'LSTM': nn.LSTM, 'GRU': nn.GRU}[rnn_type]
        self.rnn = rnn_cls(EMBED_DIM, HIDDEN_DIM, num_layers=N_LAYERS, batch_first=True)

        self.rnn_type = rnn_type
        self.fc = nn.Linear(HIDDEN_DIM, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        emb = self.embedding(x)                          # (B, T, E)
        out = self.rnn(emb)
        if self.rnn_type == 'LSTM':
            hidden = out[1][0][-1]                       # (B, H)
        else:
            hidden = out[1][-1]                          # (B, H)
        return self.sigmoid(self.fc(hidden)).squeeze(1)

print("Models defined.")

Models defined.


In [5]:
def train_model(model, loader, epochs=10, lr=0.001):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCELoss()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct = 0, 0
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            preds = model(X)
            loss  = criterion(preds, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct    += ((preds >= 0.5) == y).sum().item()

        acc = correct / len(loader.dataset)
        print(f"  Epoch {epoch:02d} | Loss: {total_loss/len(loader):.4f} | Train Acc: {acc:.4f}")


def evaluate_model(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            preds = model(X)
            all_preds.extend((preds.cpu() >= 0.5).int().numpy())
            all_labels.extend(y.int().numpy())

    print(f"  Accuracy : {accuracy_score(all_labels, all_preds):.4f}")
    print(f"  Precision: {precision_score(all_labels, all_preds):.4f}")
    print(f"  Recall   : {recall_score(all_labels, all_preds):.4f}")
    print(f"  F1-Score : {f1_score(all_labels, all_preds):.4f}")
    print(f"  Confusion Matrix:\n{confusion_matrix(all_labels, all_preds)}")


In [6]:
for rnn_type in ['RNN', 'LSTM', 'GRU']:
    print(f"\n{'='*45}")
    print(f"  {rnn_type} Model")
    print(f"{'='*45}")
    model = SentimentModel(rnn_type=rnn_type)
    train_model(model, train_loader, epochs=10)
    print(f"\n--- Test Results ({rnn_type}) ---")
    evaluate_model(model, test_loader)


  RNN Model
  Epoch 01 | Loss: 0.6968 | Train Acc: 0.5000
  Epoch 02 | Loss: 0.6890 | Train Acc: 0.5191
  Epoch 03 | Loss: 0.6803 | Train Acc: 0.5417
  Epoch 04 | Loss: 0.6936 | Train Acc: 0.5228
  Epoch 05 | Loss: 0.6883 | Train Acc: 0.5266
  Epoch 06 | Loss: 0.6791 | Train Acc: 0.5387
  Epoch 07 | Loss: 0.6674 | Train Acc: 0.5524
  Epoch 08 | Loss: 0.6538 | Train Acc: 0.5637
  Epoch 09 | Loss: 0.6392 | Train Acc: 0.5746
  Epoch 10 | Loss: 0.6320 | Train Acc: 0.5844

--- Test Results (RNN) ---
  Accuracy : 0.5037
  Precision: 0.5132
  Recall   : 0.1436
  F1-Score : 0.2244
  Confusion Matrix:
[[10797  1703]
 [10705  1795]]

  LSTM Model
  Epoch 01 | Loss: 0.6926 | Train Acc: 0.5110
  Epoch 02 | Loss: 0.6846 | Train Acc: 0.5324
  Epoch 03 | Loss: 0.6657 | Train Acc: 0.5543
  Epoch 04 | Loss: 0.6047 | Train Acc: 0.6736
  Epoch 05 | Loss: 0.5829 | Train Acc: 0.6970
  Epoch 06 | Loss: 0.6454 | Train Acc: 0.6047
  Epoch 07 | Loss: 0.6154 | Train Acc: 0.6448
  Epoch 08 | Loss: 0.4674 | Trai